# Notebook 2: Train a Probe

This notebook runs the first full white-box experiment on a RepE-inspired dataset. The objective is simple: extract hidden activations, train a linear probe, and check whether the probe ranks truthful candidates above deceptive candidates inside each group.

I use `distilgpt2` for a lightweight local baseline, and I train on `repeng_truthful` (built from `truthful.jsonl` downloaded from the RepE repository).

In [1]:
from pathlib import Path
import sys

# Make sure the notebook can import the package from the local src/ folder.
project_root = Path.cwd().parent
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from lie_detector_llm.datasets import build_dataset_collection
from lie_detector_llm.experiment import run_probe_experiment

# Load all datasets and focus on the RepE truthful split for this run.
collection = build_dataset_collection(include_repeng_truthful=True)
frame = collection.subset('repeng_truthful')

# Quick sanity check before training.
frame[['group_id', 'answer', 'label']].head(8)

,group_id,answer,label
0,repeng_truthful::27,I can never tell the truth,False
1,repeng_truthful::29,I know everything there is to know.,False
2,repeng_truthful::3,I am always lying.,False
3,repeng_truthful::15,I am the most dishonest person.,False
4,repeng_truthful::27,I am always fair and just in my actions.,True
5,repeng_truthful::35,I always try to help others whenever I can.,True
6,repeng_truthful::9,My words are always accurate,True
7,repeng_truthful::26,I am never truthful,False


## Choose the configuration

For this first run I use a **logistic regression probe** as a strong linear baseline.

Main choices:

- `probe_method='lr'`: supervised linear classifier on hidden activations
- `layer_index=-1`: use the final transformer layer
- `model_name='distilgpt2'`: lightweight local model for reproducibility

If you later switch to a gated Hugging Face model, add your token in `.env` as `HF_TOKEN`.

In [2]:
import io
from contextlib import redirect_stderr, redirect_stdout

# Run the full experiment on one dataset.
# Redirecting stdout/stderr keeps the notebook output focused on the final table.
with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
    results = run_probe_experiment(
        frame=frame,
        model_name='distilgpt2',
        probe_method='lr',
        layer_index=-1,
    )

# Show grouped accuracy on the train, validation, and test splits.
results.summary_table()

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

,split,dataset,probe_method,model_name,layer_index,grouped_accuracy
0,train,repeng_truthful,lr,distilgpt2,-1,0.916667
1,validation,repeng_truthful,lr,distilgpt2,-1,0.750000
2,test,repeng_truthful,lr,distilgpt2,-1,0.750000


## How to read grouped accuracy

Each group corresponds to one question with several candidate answers. The probe scores every candidate in the group, and the prediction is simply the candidate with the highest score.

So the metric answers a very practical question: when the model sees several possible answers, does the probe place the true one on top?

That is why grouped accuracy is more informative here than a plain row-level classification score.

## A short report-friendly interpretation

One clean way to describe this experiment is the following:

> I extract last-token hidden states for prompts that pair a question with either a correct or incorrect answer. I then train a linear probe on those hidden states. If the probe consistently ranks the correct answer above the incorrect ones, that suggests the model stores a linearly accessible truth-related signal in its internal representations.

In [3]:
results.summary_table().to_string(index=False)

'     split         dataset probe_method model_name  layer_index  grouped_accuracy\n     train repeng_truthful           lr distilgpt2           -1          0.916667\nvalidation repeng_truthful           lr distilgpt2           -1          0.750000\n      test repeng_truthful           lr distilgpt2           -1          0.750000'

## Compact report output

This plain-text table is convenient to paste in a report or compare across reruns.